# Creación BD S&P500

La idea es que del dataset con los precios, se tome el último día y se resten 1000 días, después se compare con el otro dataset de tickers para evaluar el sesgo de supervivencia. Los enlaces a dichos datasets se encuentran a continuación:

Dataset con la información de precios: https://www.kaggle.com/datasets/camnugent/sandp500?resource=download

Dataset con la información de tickers: https://github.com/fja05680/sp500

In [35]:
import pandas as pd
import os
import numpy as np


In [36]:
# ---------------------------------------------------------
# BLOQUE 1: Configuración de Rutas y Carga Inicial
# ---------------------------------------------------------
folder_path = r'../data/raw'
path_main = os.path.join(folder_path, 'all_stocks_5yr.csv')
path_components = os.path.join(folder_path, 'S&P 500 Historical Components & Changes(01-17-2026).csv')

print("Cargando dataset principal...")
df = pd.read_csv(path_main)
df['date'] = pd.to_datetime(df['date'])

# Aseguramos el orden cronológico para que el recorte de registros sea correcto
df = df.sort_values(['Name', 'date'])

# ---------------------------------------------------------
# BLOQUE 2: Filtrado por Registros (Metodología del Paper)
# ---------------------------------------------------------
# El paper utiliza 1000 días de trading (750 training + 250 trading).
# Filtramos primero para no pasarnos de febrero de 2018
df_2018 = df[df['date'].dt.year <= 2018].copy()

print("Filtrando los últimos 1000 registros de trading por cada acción...")
# Usamos groupby + tail(1000) para obtener los últimos 1000 registros de cada ticker
df_filtrado = df_2018.groupby('Name').tail(1000).copy()

# Verificación de la fecha máxima y mínima resultante
print(f"Rango de fechas en el dataset filtrado: {df_filtrado['date'].min().date()} a {df_filtrado['date'].max().date()}")

Cargando dataset principal...
Filtrando los últimos 1000 registros de trading por cada acción...
Rango de fechas en el dataset filtrado: 2014-02-03 a 2018-02-07


In [37]:
# ---------------------------------------------------------
# BLOQUE 3: Análisis de Sesgo de Supervivencia
# ---------------------------------------------------------
print("\nAnalizando componentes históricos para verificar sesgo...")
df_hist_comp = pd.read_csv(path_components)
df_hist_comp['date'] = pd.to_datetime(df_hist_comp['date'])

# Obtenemos la lista oficial del S&P 500 en la fecha final del dataset [cite: 1086]
fecha_final = df_filtrado['date'].max()
comp_en_fecha = df_hist_comp[df_hist_comp['date'] <= fecha_final].sort_values('date', ascending=False).iloc[0]

tickers_maestros = set(comp_en_fecha['tickers'].split(','))
tickers_en_dataset = set(df_filtrado['Name'].unique())

missing_stocks = tickers_maestros - tickers_en_dataset
survivors_only = tickers_en_dataset - tickers_maestros

print(f"--- RESULTADOS DEL ANÁLISIS ---")
print(f"Empresas en la Lista Maestra (Oficial): {len(tickers_maestros)}")
print(f"Empresas en tu Dataset: {len(tickers_en_dataset)}")
print(f"Empresas faltantes (Sesgo de Supervivencia): {len(missing_stocks)}")
print(f"Empresas 'extra' (Posibles deslistadas incluidas): {len(survivors_only)}")


Analizando componentes históricos para verificar sesgo...
--- RESULTADOS DEL ANÁLISIS ---
Empresas en la Lista Maestra (Oficial): 506
Empresas en tu Dataset: 505
Empresas faltantes (Sesgo de Supervivencia): 7
Empresas 'extra' (Posibles deslistadas incluidas): 6


In [38]:
# ---------------------------------------------------------
# BLOQUE 4: Almacenamiento y Verificación Final
# ---------------------------------------------------------
output_path = os.path.join(folder_path, 'sp500_ready_1000_records.csv')
df_filtrado.to_csv(output_path, index=False)

# Verificamos un ticker al azar para confirmar que tiene (o se acerca a) los 1000 registros
ejemplo_ticker = df_filtrado['Name'].iloc[0]
conteo_ejemplo = len(df_filtrado[df_filtrado['Name'] == ejemplo_ticker])
print(f"\nVerificación: La acción {ejemplo_ticker} tiene {conteo_ejemplo} registros.")
print(f"Archivo final guardado en: {output_path}")


Verificación: La acción A tiene 1000 registros.
Archivo final guardado en: ../data/raw/sp500_ready_1000_records.csv


In [39]:
# ---------------------------------------------------------
# BLOQUE 5: Identificación de Empresas por Volumen de Datos
# ---------------------------------------------------------

# 1. Contamos cuántos registros tiene cada empresa (usando el dataframe de 2018)
conteo_registros = df_2018['Name'].value_counts()

# 2. Filtramos las empresas que cumplen el requisito de 1000 registros
empresas_completas = conteo_registros[conteo_registros >= 1000]
empresas_incompletas = conteo_registros[conteo_registros < 1000]

# 3. Mostramos los resultados estadísticos
print(f"--- ANÁLISIS DE DENSIDAD DE DATOS ---")
print(f"Total de empresas analizadas: {len(conteo_registros)}")
print(f"Empresas con >= 1000 registros: {len(empresas_completas)} (Aptas para replicación total)")
print(f"Empresas con < 1000 registros: {len(empresas_incompletas)} (Historia parcial)")

# 4. Ver ejemplos de empresas que no llegan a los 1000 registros
if len(empresas_incompletas) > 0:
    print(f"\nEjemplos de empresas con pocos datos (menos de 1000):")
    # Mostramos las 5 empresas con menos registros
    print(empresas_incompletas.tail(5))

# 5. Opcional: Crear una lista de tickers "aptos" para el entrenamiento intensivo
tickers_aptos = empresas_completas.index.tolist()

--- ANÁLISIS DE DENSIDAD DE DATOS ---
Total de empresas analizadas: 505
Empresas con >= 1000 registros: 483 (Aptas para replicación total)
Empresas con < 1000 registros: 22 (Historia parcial)

Ejemplos de empresas con pocos datos (menos de 1000):
Name
DXC     215
BHGE    152
BHF     143
DWDP    109
APTV     44
Name: count, dtype: int64


### Avoiding survivor bias

To eliminate survivor bias, we first obtain all month-end constituent lists 
for the S&P 500 from January 2013 to December 2018. We consolidate these lists 
into a single binary matrix, indicating whether each stock is an index 
constituent in the subsequent month. In this way, we are able to approximately 
reproduce the S&P 500 composition at any point in time within this period. 
In a second step, for all stocks that have been constituents of the index at 
any time during this interval, we download daily total return indices covering 
the same time frame.

In [40]:
df_hist_comp.head()

,date,tickers
0,1996-01-02,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
1,1996-01-03,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
2,1996-01-04,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
3,1996-01-10,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
4,1996-01-11,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."


In [41]:
df_hist_comp.tail()

,date,tickers
2700,2025-11-11,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP..."
2701,2025-11-28,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP..."
2702,2025-12-11,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP..."
2703,2025-12-22,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP..."
2704,2026-01-14,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP..."


In [42]:
df_hist_comp.dtypes

date       datetime64[us]
tickers               str
dtype: object

In [43]:
df.head()
first_date = df.iloc[0]['date']
print(first_date)

last_date = df.iloc[-1]['date']
print(last_date)

2013-02-08 00:00:00
2018-02-07 00:00:00


In [44]:
# Filtrar por fechas
df_hist_comp = df_hist_comp[(df_hist_comp['date'] >= first_date) & (df_hist_comp['date'] <= last_date)].copy()

# Extraer año y mes
df_hist_comp['year'] = df_hist_comp['date'].dt.year
df_hist_comp['month'] = df_hist_comp['date'].dt.month

# Tomar último día de cada mes
df_month_end = df_hist_comp.groupby(['year', 'month']).tail(1)

# Eliminar la columna 'date'
df_month_end = df_month_end.drop(columns=['date'])

# Reordenar columnas para que year y month queden primero
cols = ['year', 'month'] + [c for c in df_month_end.columns if c not in ['year', 'month']]
df_month_end = df_month_end[cols]

df_month_end = df_month_end.reset_index(drop=True)

# Revisar resultado
print(df_month_end.head())
print("...")
print(df_month_end.tail())

   year  month                                            tickers
0  2013      2  A,AABA,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,ADP,...
1  2013      3  A,AABA,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,ADP,...
2  2013      4  A,AABA,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,ADP,...
3  2013      5  A,AABA,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,ADP,...
4  2013      6  A,AABA,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,ADP,...
...
    year  month                                            tickers
56  2017     10  A,AAL,AAP,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,A...
57  2017     11  A,AAL,AAP,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,A...
58  2017     12  A,AAL,AAP,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,A...
59  2018      1  A,AAL,AAP,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,A...
60  2018      2  A,AAL,AAP,AAPL,ABBV,ABC,ABT,ACN,ADBE,ADI,ADM,A...


In [45]:
all_tickers = set()

for tickers_str in df_hist_comp['tickers']:
    for t in tickers_str.split(','):
        all_tickers.add(t.strip())
all_tickers = sorted(all_tickers)

ticker_dict = {ticker: idx for idx, ticker in enumerate(all_tickers)}


n_months = len(df_month_end)
n_tickers = len(all_tickers)

# matriz de ceros
binary_matrix = np.zeros((n_months, n_tickers), dtype=int)

for i, row in df_month_end.iterrows():
    tickers_in_row = row['tickers'].split(',')  # separa los tickers
    for t in tickers_in_row:
        t = t.strip()  # quitar espacios
        if t in ticker_dict:  # seguridad
            j = ticker_dict[t]  # columna correspondiente
            binary_matrix[i, j] = 1


binary_df = pd.DataFrame(binary_matrix, columns=all_tickers)
binary_df.insert(0, 'year', df_month_end['year'])
binary_df.insert(1, 'month', df_month_end['month'])

print(binary_df.head())

   year  month  A  AABA  AAL  AAP  AAPL  ABBV  ABC  ABT  ...  XL  XLNX  XOM  \
0  2013      2  1     1    0    0     1     1    1    1  ...   1     1    1   
1  2013      3  1     1    0    0     1     1    1    1  ...   1     1    1   
2  2013      4  1     1    0    0     1     1    1    1  ...   1     1    1   
3  2013      5  1     1    0    0     1     1    1    1  ...   1     1    1   
4  2013      6  1     1    0    0     1     1    1    1  ...   1     1    1   

   XRAY  XRX  XYL  YUM  ZBH  ZION  ZTS  
0     1    1    1    1    1     1    0  
1     1    1    1    1    1     1    0  
2     1    1    1    1    1     1    0  
3     1    1    1    1    1     1    0  
4     1    1    1    1    1     1    1  

[5 rows x 619 columns]


### Create rolling windows

#### Paso 1: Definir los bloques de 1000 dias

In [46]:
df_rolling_windows = df  # tu DataFrame de precios diarios


all_dates = pd.bdate_range(start=df['date'].min(), end=df['date'].max())

training_days = 750
trading_days = 250
study_period_days = training_days + trading_days

start_idx = 0
study_periods = []

while start_idx + study_period_days <= len(all_dates):
    period_dates = all_dates[start_idx:start_idx + study_period_days]
    training_dates = period_dates[:training_days]
    trading_dates = period_dates[training_days:]
    
    study_periods.append({
        'training_dates': training_dates,
        'trading_dates': trading_dates
    })
    
    start_idx += trading_days  # rolling cada 250 días

print(len(study_periods))

2


Solo tendremos dos periodos de estudio

#### Paso 2: Filtrar tickers usando la matriz binaria

In [47]:
def get_tickers_for_period(binary_df, last_train_date):
    year = last_train_date.year
    month = last_train_date.month
    
    row = binary_df[
        (binary_df['year'] == year) &
        (binary_df['month'] == month)
    ]
    
    if row.empty:
        return []
    
    tickers_series = row.drop(columns=['year', 'month']).iloc[0]
    tickers = tickers_series[tickers_series == 1].index.tolist()
    
    return tickers


In [48]:
study_periods_data = []

for i, period in enumerate(study_periods):
    
    last_train_date = pd.to_datetime(period['training_dates'][-1])
    
    tickers = get_tickers_for_period(binary_df, last_train_date)
    
    # 🔹 Filtrar training
    df_training = df_rolling_windows[
        (df['Name'].isin(tickers)) &
        (df['date'].isin(period['training_dates']))
    ].copy()
    
    # 🔹 Filtrar trading
    df_trading = df_rolling_windows[
        (df['Name'].isin(tickers)) &
        (df['date'].isin(period['trading_dates']))
    ].copy()
    
    # Guardar todo
    study_periods_data.append({
        'period_id': i,
        'tickers': tickers,
        'n_i': len(tickers),
        'train_df': df_training,
        'test_df': df_trading,
        'train_start': period['training_dates'][0],
        'train_end': period['training_dates'][-1],
        'test_start': period['trading_dates'][0],
        'test_end': period['trading_dates'][-1]
    })



In [49]:
for sp in study_periods_data:
    print(f"\nStudy Period {sp['period_id']}")
    print(f"n_i: {sp['n_i']}")
    print(f"Train: {sp['train_start']} → {sp['train_end']}")
    print(f"Test: {sp['test_start']} → {sp['test_end']}")
    print(f"#Train rows: {len(sp['train_df'])}")
    print(f"#Test rows: {len(sp['test_df'])}")


Study Period 0
n_i: 502
Train: 2013-02-08 00:00:00 → 2015-12-24 00:00:00
Test: 2015-12-25 00:00:00 → 2016-12-08 00:00:00
#Train rows: 313543
#Test rows: 106039

Study Period 1
n_i: 506
Train: 2014-01-24 00:00:00 → 2016-12-08 00:00:00
Test: 2016-12-09 00:00:00 → 2017-11-23 00:00:00
#Train rows: 335031
#Test rows: 113130


In [ ]:

import numpy as np

def calculate_returns(df, price_col="price"):
    """
    Simula retornos diarios (m=1) para cada fila del DataFrame
    y los guarda en una nueva columna llamada 'return'.

    Por ahora NO usa el precio real, sino valores aleatorios
    que emulan retornos diarios (pequeños cambios alrededor de 0).
    """

    # Generar retornos aleatorios tipo mercado (~media 0, baja varianza)
    returns = np.random.normal(loc=0, scale=0.01, size=len(df))

    # Agregar columna al DataFrame
    df["return"] = returns

    return df

def create_sequences_for_stock_s(df):
    df = calculate_returns(df)
    sequences_for_ticker = []

    window_size = 240

    for i in range(window_size, len(df)):
        # tomar ventana de 240 retornos
        window = df.iloc[i - window_size:i]["return"].values
        
        # opcional: reshape para LSTM (240, 1)
        window = window.reshape(-1, 1)
        sequences_for_ticker.append(window)

    return sequences_for_ticker
         

def build_sequences_dataset_for_study_period(sp):
    sequences = []
    tickers = sp.get('tickers', [])
    train_df = sp.get('train_df')
    test_df = sp.get('test_df')
    
    for t in tickers:
        train_mask = train_df['Name'] == t
        test_mask = test_df['Name'] == t

        # filtrar dataframes correctamente
        train_data = train_df[train_mask]
        test_data = test_df[test_mask]

        # unir datos
        final_df_for_ticker = pd.concat([train_data, test_data]).sort_index()

        sequences_for_ticker = create_sequences_for_stock_s(final_df_for_ticker)
        sequences.append(sequences_for_ticker)

    return sequences


for sp in study_periods_data:
    sequences_dataset_built = build_sequences_dataset_for_study_period(sp)
    sp["sequences_dataset"] = sequences_dataset_built
    
    
    

In [ ]:
def debug_sequences(sp, max_tickers=5, max_sequences=2):
    sequences = sp.get("sequences_dataset", [])
    tickers = sp.get("tickers", [])

    print(f"\nStudy Period {sp['period_id']}")
    print(f"Total tickers: {len(sequences)}")

    for i, (ticker, ticker_sequences) in enumerate(zip(tickers, sequences)):
        if i >= max_tickers:
            break

        print(f"\nTicker: {ticker}")
        print(f"Total sequences: {len(ticker_sequences)}")

        for j, seq in enumerate(ticker_sequences[:max_sequences]):
            print(f"  Seq {j} shape: {seq.shape}")
            print(f"  First 3 values: {seq[:3].flatten()}")

debug_sequences(study_periods_data[0])